# Safe Momentum: live paper-trading notebook

Multi-asset dual momentum (see `research/safe_research.ipynb`).

**Run All** once a month after month end. With TWS (or IB Gateway) open, account value and positions are read automatically through a **read-only** API session; orders are entered manually.

Credentials and the account id are read from `ibkr_keys.env` (not in the repository, see `ibkr_keys.env.example`).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.append(os.path.join("..", "src"))
from data import download_prices, prepare_monthly
from strategies import trend_momentum_signal, correlation_filter, momentum_weights
from execution import apply_overlay, rebalance_orders
from ibkr import load_keys, read_portfolio
from factsheet import build_factsheet

STRATEGY   = "SAFE"
DATA_DIR   = os.path.join("..", "data")
KEYS       = load_keys(os.path.join("..", "ibkr_keys.env"))
ACCOUNT    = KEYS.get("IBKR_ACCOUNT_SAFE", "")
USE_IBKR   = True     # False: type positions manually in section 3
IBKR_PORT  = 7497     # 7497 TWS paper | 7496 TWS live | 4002 Gateway paper | 4001 Gateway live
CLIENT_ID  = 11       # any integer, different for each notebook

## 1. Data and model weights

In [ ]:
UNIVERSE = ["SPY", "QQQ", "IWM", "VWO", "GLD", "SHV", "IEF", "TLT", "TIP", "LQD"]

prices = download_prices(UNIVERSE, years=20, cache_path=os.path.join(DATA_DIR, "prices_safe.csv"))
md = prepare_monthly(prices, sma_window=200)
selected = correlation_filter(trend_momentum_signal(md), md.mom_3m, md.daily_returns,
                              max_assets=5, corr_threshold=0.80)
weights = momentum_weights(selected, md.mom_3m, base_cap=0.25, turbo_cap=0.40, turbo_trigger=0.05)

## 2. Targets (optional discretionary overlay)
Tweaks must net to zero. Every deviation from the model is logged in the journal line.

In [ ]:
tweaks = {t: 0.0 for t in UNIVERSE}
journal = "Model weights, no discretionary change."

targets = apply_overlay(weights.iloc[-1], tweaks, journal)

## 3. Current positions

In [ ]:
# Backtest ticker -> EUR-listed UCITS ETF actually traded
UCITS_MAP = {
    "SPY": {"ticker": "SXR8", "isin": "IE00B5BMR087", "desc": "iShares Core S&P 500"},
    "QQQ": {"ticker": "SXRV", "isin": "IE00B53SZB19", "desc": "iShares Nasdaq 100"},
    "IWM": {"ticker": "ZPRR", "isin": "IE00BJ38QD84", "desc": "SPDR Russell 2000"},
    "VWO": {"ticker": "EMIM", "isin": "IE00BKM4GZ66", "desc": "iShares Core MSCI EM IMI"},
    "GLD": {"ticker": "SGLD", "isin": "IE00B579F325", "desc": "Invesco Physical Gold"},
    "TLT": {"ticker": "IS04", "isin": "IE00BSKRJZ44", "desc": "iShares USD Treasury 20+yr"},
    "IEF": {"ticker": "BTMA", "isin": "IE00B3VWN179", "desc": "iShares USD Treasury 7-10yr"},
    "SHV": {"ticker": "IBCC", "isin": "IE00BGSF1X88", "desc": "iShares USD Treasury 0-1yr"},
    "TIP": {"ticker": "TIUP", "isin": "LU1452600270", "desc": "Amundi US TIPS"},
    "LQD": {"ticker": "LQDA", "isin": "IE00B3F81R35", "desc": "iShares USD Corp Bond IG"},
}

if USE_IBKR:
    account_value, holdings = read_portfolio(ACCOUNT, UCITS_MAP, port=IBKR_PORT, client_id=CLIENT_ID)
else:
    account_value = 0.0                                  # e.g. 105480.00
    holdings = {k: 0.0 for k in UCITS_MAP}              # current value in EUR per ticker

## 4. Order list

In [ ]:
orders = rebalance_orders(targets, holdings, account_value, UCITS_MAP, min_trade=50.0)

## 5. Factsheet

In [ ]:
START_DATE     = "2026-04-02"            # start of the paper track record
FUND_NAME      = "Safe Momentum Portfolio"
UNIVERSE_LABEL = "10 ETFs (multi-asset)"
ASSET_CLASS = {"SPY": "Equity", "QQQ": "Equity", "IWM": "Equity", "VWO": "Equity", "GLD": "Gold",
               "TLT": "Govt bonds", "IEF": "Govt bonds", "TIP": "Govt bonds",
               "LQD": "Corporate bonds", "SHV": "Cash-like (T-bills)"}
STRATEGY_TEXT = """
Systematic multi-asset momentum strategy designed for capital preservation. The universe covers
10 ETFs: US large, tech and small-cap equities, emerging markets, gold, US Treasuries across the
curve, inflation-linked bonds and investment-grade credit. The portfolio is rebalanced monthly.
An asset is eligible only if its price is above its 200-day moving average and its 3-month return
is positive (dual momentum). Among eligible assets, pairs with 6-month correlation above 0.80 are
filtered, keeping the one with stronger momentum. Up to 5 assets are held, weighted by 3-month
momentum with a 25% cap per position (40% when the leading asset gains more than 5% over 3 months).
Any unallocated weight is held in 0-1Y US Treasuries, the strategy's cash proxy. When no asset is
eligible the portfolio moves entirely to T-bills. Implemented with EUR-listed UCITS ETFs.
"""

build_factsheet(
    strategy=STRATEGY, account=ACCOUNT, account_value=account_value, holdings=holdings,
    targets=targets, model_weights=weights, ucits_map=UCITS_MAP, asset_class=ASSET_CLASS,
    fund_name=FUND_NAME, universe=UNIVERSE_LABEL, strategy_text=STRATEGY_TEXT, start_date=START_DATE,
    flex_token=KEYS.get("FLEX_TOKEN_SAFE", ""),
    flex_query_id=KEYS.get("FLEX_QUERY_ID_SAFE", ""),
    use_ibkr=USE_IBKR, data_dir=DATA_DIR, out_dir=os.path.join("..", "factsheets"),
);